In [92]:
import numpy as np
import fury

fury.data.fetch_viz_textures();

Dataset is already in place. If you want to fetch it again please first remove the folder /home/rio/.fury/textures 


In [93]:
def acceleration(positions, masses, softening=SOFTENING):
    """
    Compute gravitational accelerations on each body (2D).
    Args:
        positions: shape (N, 2), array of 2D positions for each body 
        masses: shape (N,), array of masses for each body
        softening: to avoid numerical instability when bodies are very close
    Returns:
        2D array of accelerations for each body
    """
    diff = positions[np.newaxis, :, :] - positions[:, np.newaxis, :]
    dist_sq = np.sum(diff**2, axis=-1) + softening**2
    inv_dist_cube = dist_sq**(-1.5)
    np.fill_diagonal(inv_dist_cube, 0.0)

    acc = G * np.sum(masses[np.newaxis, :, np.newaxis] *
                     diff * inv_dist_cube[:, :, np.newaxis], axis=1)
    return acc

def leapfrog_step(positions, velocities, masses, dt):
    """
    Perform one integration step using Leapfrog method. 
    """
    positions_new = positions + dt * velocities
    acc_new = acceleration(positions_new, masses)
    velocities_new = velocities + dt * acc_new
    return positions_new, velocities_new

In [124]:
# 1. Constants and setup
# Gravitational constant in AU^3 / (yr^2 * M_sun)
# Chosen so that a=1 AU, M=1 M_sun gives P=1 yr
G = 4 * np.pi**2

S0 = 1361.0          # Solar constant (W/m^2)

M_SUN = 1.0
M_EARTH = 3.003e-6   # in solar masses
M_JUPITER = 9.545e-4 # in solar masses

SUN_RADIUS = 4.6492e-3 # AU

EARTH_R0 = 1.0       # AU
EARTH_V0 = 2 * np.pi # AU/yr (circular)
EARTH_V0_ECCENTRIC = 2 * np.pi * 1.01
EARTH_RADIUS = 4.2494e-5 # AU

JUPITER_R0 = 5.2     # AU
JUPITER_V0 = 2 * np.pi / np.sqrt(JUPITER_R0)
JUPITER_RADIUS = 4.7789e-4 # AU

# sun and solar system is too big
SUN_RADIUS *= 1e2
EARTH_RADIUS *= 1e3
JUPITER_RADIUS *= 1e3

SOFTENING = 0.01     # AU (1% of Earth–Sun distance)

phi_65N = np.radians(65.0)

bodies='sun_earth_jupiter'
eccentric_init=False
dt = 0.02

if bodies == 'sun_earth':
    masses = np.array([M_SUN, M_EARTH])
    positions = np.array([
        [0.0, 0.0],
        [EARTH_R0, 0.0],
    ])
    v_earth = np.array([0.0, EARTH_V0])
    v_sun = -(M_EARTH / M_SUN) * v_earth
    velocities = np.array([
        v_sun,
        v_earth,
    ])
    radii = np.array([SUN_RADIUS, EARTH_RADIUS])
else:  # 'sun_earth_jupiter'
    masses = np.array([M_SUN, M_EARTH, M_JUPITER])
    positions = np.array([
        [0.0, 0.0],
        [EARTH_R0, 0.0],
        [JUPITER_R0, 0.0],
    ])
    radii = np.array([SUN_RADIUS, EARTH_RADIUS, JUPITER_RADIUS])
    if eccentric_init:
        velocities = np.array([
            [0.0, 0.0],
            [0.0, EARTH_V0_ECCENTRIC],
            [0.0, JUPITER_V0],
        ])
    else:
        velocities = np.array([
            [0.0, 0.0],
            [0.0, EARTH_V0],
            [0.0, JUPITER_V0],
        ])

# Center-of-mass correction to ensure total momentum is zero
total_mass = np.sum(masses)
x_cm = np.sum(positions * masses[:, None], axis=0) / total_mass
v_cm = np.sum(velocities * masses[:, None], axis=0) / total_mass

pos = positions.copy()
vel = velocities.copy()

pos -= x_cm
vel -= v_cm

In [125]:
def update(_obj, _event):
    global pos, vel
    pos, vel = leapfrog_step(pos, vel, masses, dt)
    for i, p in enumerate(pos):
        actors[i].SetPosition(*p, 0)
    showm.render()

scene = fury.window.Scene()
showm = fury.window.ShowManager(scene, size=(1024,768))

filenames = ["8k_sun.jpg", "1_earth_8k.jpg", "jupiter.jpg"]
textures = [fury.io.load_image(fury.data.read_viz_textures(filenames[i])) for i,_ in enumerate(positions)]
actors = [fury.actor.texture_on_sphere(texture) for texture in textures]

for i, actor in enumerate(actors):
    scene.add(actor)
    actor.SetPosition(*positions[i], 0)
    actor.SetScale(radii[i])

scene.set_camera(position=(0, 0, 25))
showm.add_iren_callback(lambda _obj, _event: showm.scene.reset_clipping_range(), event='MouseMoveEvent')

showm.add_timer_callback(True, 16, update)
showm.start()